## Ejemplo basico de entrada y salida

In [2]:
from openai import AzureOpenAI # importamos el cliente especifico de Azure


# utilizamos las credenciales para conectarnos al modelo LLM dentro del codigo sin usar el .env
AZURE_OPENAI_ENDPOINT = "https://onboarding-practicante-azure-openai.openai.azure.com/"
AZURE_OPENAI_API_KEY = "2w7J63JY3f9rQiehNVJrO4ksnKGJ47Cv2gtDnkN80fhacwDz9qbPJQQJ99BLACYeBjFXJ3w3AAABACOGqeD0"
AZURE_DEPLOYMENT_NAME = "gpt-4o-mini"
AZURE_API_VERSION = "2025-04-01-preview"

# inicializamos Azure
client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_API_VERSION
    )

# Almacenamos las respuestas de la API
response = client.chat.completions.create(
    # usamos la variable del modelo que procesara la informacion
    model=AZURE_DEPLOYMENT_NAME,
    # Historial de la conversacion enviada al modelo
    messages=[{"role": "user", "content": "Hola, ¿quien eres?"}],
)

# imprimimos la primer respuesta con choices[0]
print(response.choices[0].message.content)



¡Hola! Soy un modelo de lenguaje de inteligencia artificial creado por OpenAI. Estoy aquí para ayudarte a responder preguntas, ofrecer información o simplemente conversar. ¿En qué puedo ayudarte hoy?


## Flujo básico con Langchain: LLMChain con PromptTemplate. 

In [3]:
import getpass
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

# Esquemapara la salida
class Persona(BaseModel):
    name: str = Field(description="El nombre completo de la persona.")
    height: str = Field(description="La altura de la persona, incluyendo unidades.")
    hair_color: str = Field(description="El color del cabello de la persona.")

# carga de variables del sistema
load_dotenv()

# verificamos 
if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter API KEY for Azure: ")

# iniciamos el LLM de Azure
llm = AzureChatOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

# Creamos el objeto LLM que obliga a la salida a seguir el esquema 'Persona'
structured_llm = llm.with_structured_output(Persona)

# definimos el prompt
template = """
Eres un extractor de datos experto.
Analiza el siguiente texto y extrae las propiedades definidas en el esquema JSON.

Texto a analizar:
{text}
"""

# creamos el objeto con PromptTemplate
prompt_template = PromptTemplate(
    input_variables=["text"],
    template=template,
)

# Datos de entrada
text = "Alan Smith is 6 feet tall and has blond hair."

# Invocamos el Prompt (inserta el texto en la plantilla)
prompt = prompt_template.invoke({"text": text})

# Ejecutamos el modelo estructurado
resultado = structured_llm.invoke(prompt)

# Imprimimos
print(resultado)
print(resultado.model_dump_json(indent=2))

name='Alan Smith' height='6 feet' hair_color='blond'
{
  "name": "Alan Smith",
  "height": "6 feet",
  "hair_color": "blond"
}


In [6]:
# Definición de metadatos del modelo para referencia y trazabilidad
model_name = "gpt-4o-mini"
deployment = "gpt-4o-mini"

# Captura interactiva de inputs del usuario que se utilizarán como variables de prompt.
# Esto asegura que el contenido generado sea específico y contextual.
animal = input("Escribe el nombre del animal: ")
producto = input("Escribe el nombre del producto: ")

# Creación del objeto PromptTemplate.
plantilla = PromptTemplate(
    # 'input_variables' define las claves esperadas para el relleno del template.
    input_variables=["animal", "producto"],
    # 'template' es la instrucción central.
    # para inyectar dinámicamente las entradas del usuario.
    template="Escribe una descripcion para vender un {producto} para {animal}"
)

# Creación del pipeline de ejecución
# Esto encadena la plantilla (PromptTemplate) directamente al modelo LLM (llm),
# donde 'llm' es la instancia de AzureChatOpenAI previamente configurada.
# El flujo es: Input -> Plantilla -> LLM de Azure -> Output.
cadena = plantilla | llm

# Invocación de la cadena con el método estandarizado .invoke().
# La ejecución realiza el reemplazo de variables en el template y envía la solicitud a Azure.
resultado = cadena.invoke({"animal": animal, "producto": producto})

respuesta = resultado.content

nombre_archivo = "publicidad.txt"

with open (nombre_archivo, "w", encoding="utf-8") as archivo:
    archivo.write(respuesta)

print(f"Respuesta del Modelo 1 guardada exitosamente en: {nombre_archivo}")

Respuesta del Modelo 1 guardada exitosamente en: publicidad.txt
